# 07 动量情绪联合策略

## 本课学习目标

- A. 量化金融主线：投资组合（Portfolio）、交易费用（Transaction Cost）和滑点（Slippage）
零基础解释：组合是一篮子资产，费用和滑点会降低回测收益。
- B. 大语言模型主线：大语言模型提供商（LLM Provider）
零基础解释：本课让 Mock Provider 只提供情绪因子，不让模型直接交易。
- C. 两条线如何连接：把市场数据和新闻文本转成可检查的表格信号。
- D. 可运行实验：生成动量情绪联合权重，运行独立教学回测并输出指标。
- E. 结果解释：观察表格、图表和结构化输出。
- F. 常见错误：把回测收益当成未来收益、把 Mock 当成真实模型。
- G. 课后练习：修改一个参数并重新运行。
- H. 本课术语表：见本课各小节。

## 本课最终输出

一个离线实验输出，不联网、不调用真实模型、不产生真实订单。

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "learning").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
DATA = ROOT / "learning" / "data"


## 可运行实验

下面代码只读取 `learning/data` 下的合成数据。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from learning.src.market_data import load_price_data
from learning.src.momentum_factor import compute_momentum, rank_momentum
from learning.src.sentiment_factor import classify_news, daily_sentiment_factor
from learning.src.time_alignment import monthly_signal_schedule
from learning.src.mini_backtest import run_backtest
from learning.src.financial_metrics import simple_returns, annualized_return, annualized_volatility, maximum_drawdown, sharpe_ratio
prices = load_price_data(DATA / "sample_prices.csv")
news = classify_news(pd.read_csv(DATA / "sample_news.csv"))
schedule = monthly_signal_schedule(prices).head(8)
mom = compute_momentum(prices, 20)
targets = []
for _, s in schedule.iterrows():
    m = rank_momentum(mom, s["signal_date"], 20)
    sent = daily_sentiment_factor(news, s["signal_timestamp"]).groupby("ticker")["sentiment_score"].mean()
    m["sentiment_score"] = m["ticker"].map(sent).fillna(0.0)
    m["combined_score"] = 0.7 * m["momentum_rank_score"] + 0.3 * ((m["sentiment_score"] + 1) / 2)
    for ticker in m.nlargest(2, "combined_score")["ticker"]:
        targets.append({"execution_date": s["execution_date"], "ticker": ticker, "weight": 0.5})
targets = pd.DataFrame(targets)
result = run_backtest(prices, targets, transaction_cost=0.001, slippage=0.0005)
eq = result.equity_curve
rets = simple_returns(eq["equity"])
metrics = {
    "cumulative": eq["equity"].iloc[-1] / eq["equity"].iloc[0] - 1,
    "annual_return": annualized_return(rets),
    "annual_volatility": annualized_volatility(rets),
    "max_drawdown": maximum_drawdown(eq["equity"]),
    "sharpe": sharpe_ratio(rets),
    "turnover": result.total_turnover,
    "trades": len(result.trades),
}
display(pd.DataFrame([metrics]))
eq.set_index("date")["equity"].plot(title="Teaching strategy equity")
plt.show()

## 结尾总结

你现在应该理解：量化数据和文本模型输出都必须被结构化、校验并按时间对齐。

哪些结果不能解释为策略一定赚钱：任何图表和收益数字都只是合成数据上的教学结果。

本课使用了哪些英文专业词：Large Language Model, Prompt, Structured Output, Backtesting, Factor, Return, Risk。

下一课与本课有什么关系：下一课会在本课结果上继续增加一个新量化概念和一个新 LLM 概念。